In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
import pandas as pd
import datetime as dt

from rockyclickup.wrapper import Session as cu_session
from rockyclickup.utils import response_to_dataframe as cu_response_to_dataframe
from rockyclickup.database_interface import get_clickup_users

from rockyelevate.wrapper import Session as cu_session
from rockyelevate.utils import response_to_dataframe as elv_response_to_dataframe

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from utils.clickup import fix_clickup_date, add_client_id_to_plan_df
from utils.collector import (
    get_all_elv_organizations,
    get_all_elv_plans,
    get_all_cu_clients,
    get_all_cu_plans,
)


In [ ]:
all_elevate_orgs = get_all_elv_organizations()
all_clickup_clients = get_all_cu_clients()

In [ ]:
complex_hras = pd.read_csv("HP_Set_up_1759160026358.csv")
complex_hras = complex_hras.rename(
    columns={
        "Partner": "partner",
        "Employer Name": "employer_name",
        "Employer Sys ID": "organization_id",
        "Employer ID": "rmrcode",
        "Health Plan Code": "health_plan_code",
        "Coverage Code": "coverage_tier",
        "HRA Amount": "election_amount",
    }
)

In [ ]:
complex_hra_org_ids = complex_hras['organization_id'].unique()
all_elevate_orgs['complex_hra'] = all_elevate_orgs['organization_id'].apply(lambda x: x in complex_hra_org_ids)
all_elevate_orgs['complex_hra'] = all_elevate_orgs['complex_hra'].astype(str)
all_elevate_orgs['complex_hra'] = all_elevate_orgs['complex_hra'].replace("False", "")

In [ ]:
[c for c in all_clickup_clients.columns if "manager" in c]

In [ ]:

client_id_map = {
    r.get("rmrcode"): r.get("client_id")
    for _, r in all_clickup_clients.iterrows()
}


client_id_from_name_map = {
    r.get("client_name"): r.get("client_id")
    for _, r in all_clickup_clients.iterrows()
}

client_id_from_org_map = {
    r.get("cu_client_elv_id"): r.get("client_id")
    for _,r in all_clickup_clients.iterrows()
}

am_map = {
    r.get("rmrcode"): r.get("cu_client_account_manager")
    for _, r in all_clickup_clients.iterrows()
}


# add clickup client card task id
all_elevate_orgs['employer_clickup_id'] = all_elevate_orgs['rmrcode'].map(client_id_map)
all_elevate_orgs['client_id_from_name'] = all_elevate_orgs['organization_name'].map(client_id_from_name_map)
all_elevate_orgs['client_id_from_org'] = all_elevate_orgs['organization_id'].map(client_id_from_org_map)


# fill na with other methods
all_elevate_orgs['employer_clickup_id'].fillna(all_elevate_orgs['client_id_from_name'], inplace=True)
all_elevate_orgs['employer_clickup_id'].fillna(all_elevate_orgs['client_id_from_org'], inplace=True)


# add clickup clinet cart account manager
all_elevate_orgs['account_manager'] = all_elevate_orgs['rmrcode'].map(am_map)


In [ ]:
clickup_users = get_clickup_users()

cu_user_map = {
    u.get("id"): u.get("username")
    for u in clickup_users
}

all_elevate_orgs['account_manager'] = all_elevate_orgs['account_manager'].apply(lambda x: x if isinstance(x, list) else [])
all_elevate_orgs['account_manager'] = all_elevate_orgs['account_manager'].apply(lambda x: [cu_user_map.get(y, "Deactivated") for y in x])




In [ ]:
all_elevate_orgs = all_elevate_orgs.sort_values(by="account_manager")

company_df = all_elevate_orgs[all_elevate_orgs['type'] == "COMPANY"]

renamed_df = company_df.rename(columns={
        "organization_id": "employer_elv_id",
        "rmrcode": "employer_id",
        "organization_name": "employer_name",

    }
)

necessary_columns = [
    "employer_clickup_id",
    "employer_elv_id",
    "employer_id",
    "employer_name",
    "account_manager",
    "complex_hra"
]

display(renamed_df[necessary_columns])

renamed_df.to_csv("complex_hra_analysis.csv")

In [ ]:
company_df[company_df['employer_clickup_id'].isna()][[
    "employer_clickup_id",
    "organization_id",
    "rmrcode",
    "organization_name",
    "account_manager",
    "complex_hra"
]]